# 队员 A — 主表特征工程
## application_train.csv 全流程: A1 缺失值 → A2 异常值 → A3 编码 → A4 时间 → A5 衍生

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
from src.data.preprocess_application import (
    fill_missing, treat_outliers, convert_days_features,
    engineer_features, encode_categorical, analyze_missing,
    process_application_table, merge_with_features,
)
import warnings
warnings.filterwarnings('ignore')

## 1. 加载数据

In [ ]:
train = pd.read_csv('../data/raw/application_train.csv')
test = pd.read_csv('../data/raw/application_test.csv')
print(f'Train: {train.shape}')
print(f'Test:  {test.shape}')
train.head(3)

## 2. A1 缺失值分析 & 填充

In [ ]:
missing = analyze_missing(train)
train = fill_missing(train)
test = fill_missing(test)

## 3. A2 异常值处理

In [ ]:
# DAYS_EMPLOYED = 365243 → 无业标记
train = treat_outliers(train)
print(f"无业人数: {train['FLAG_UNEMPLOYED'].sum()}")
print(f"收入范围: {train['AMT_INCOME_TOTAL'].min():.0f} ~ {train['AMT_INCOME_TOTAL'].max():.0f}")

## 4. A4 时间特征转换

In [ ]:
train = convert_days_features(train)
train[['AGE_YEARS', 'YEARS_EMPLOYED', 'YEARS_REGISTRATION']].describe()

## 5. A5 特征衍生

In [ ]:
train = engineer_features(train)
print(f'衍生后维度: {train.shape[1]}')
# 查看部分新特征
new_cols = [c for c in train.columns if c not in ['SK_ID_CURR']][-20:]
train[new_cols].head(3)

## 6. A3 类别特征编码

In [ ]:
train = encode_categorical(train)
print(f'编码后维度: {train.shape[1]}')
for col in train.select_dtypes(include=['object', 'category']).columns:
    print(f'  WARNING: 未编码列: {col}')

## 7. 保存处理后的数据

In [ ]:
train.to_csv('../data/processed/processed_application_train.csv', index=False)
print(f'已保存: processed_application_train.csv ({train.shape})')

## 8. (可选) A6: 合并队员 B/C 特征
当 B 和 C 成员完成特征工程后，运行此步骤合并为最终宽表。

In [ ]:
# 示例: 当 B、C 产出特征表后取消注释运行
# final = merge_with_features(
#     '../data/processed/processed_application_train.csv',
#     [
#         ('../data/processed/features_bureau.csv', '队员B-征信局'),
#         ('../data/processed/features_internal.csv', '队员C-内部流水'),
#     ]
# )